In [6]:
!pip install requests 

In [7]:
import requests 
from bs4 import BeautifulSoup
import re 
from collections import Counter

In [8]:
def get_website_content(url):
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    return response.text

In [9]:
def simple_sentence_split(text):
    return re.split(r'(?<=[.!?]) +', text)

In [27]:
def summarize_text(text, num_sentences=3):
    sentences = simple_sentence_split(text)
    if len(sentences) <= num_sentences:
        return " ".join(sentences)

    words = re.findall(r'\w+', text.lower())
    word_freq = Counter(words)

    sentence_scores = {}
    for sentence in sentences:
        for word in re.findall(r'\w+', sentence.lower()):
            sentence_scores[sentence] = sentence_scores.get(sentence, 0) + word_freq[word]

    top_sentences = sorted(sentence_scores, key=sentence_scores.get, reverse=True)[:num_sentences]
    return " ".join(top_sentences)

In [2]:
def find_cheapest_product(soup):
    
    if not isinstance(soup, BeautifulSoup):
        return "Nije unet dobar tekst. Očekuje se BeautifulSoup objekat."

    products = []
    price_patterns = re.compile(r'(\d+[.,]?\d*)\s?(RSD|rsd|€|EUR|eur)')

    for tag in soup.find_all(string=price_patterns):
        match = price_patterns.search(tag)
        if match:
            try:
                price = float(match.group(1).replace(",", "."))
                product_name = tag.parent.get_text(strip=True)
                products.append((product_name, price))
            except:
                pass

    if not products:
        return None

    return min(products, key=lambda x: x[1])

In [35]:
def analyze_website(url):
    try:
        html = get_website_content(url)
        soup = BeautifulSoup(html, "html.parser")
        page_text = soup.get_text(separator=" ", strip=True)


        cheapest_product = find_cheapest_product(soup)

        if cheapest_product:
            print("🛒 Izgleda da je ovo web shop.")
            print(f"Najjeftiniji proizvod: {cheapest_product[0]}")
            print(f"Cena: {cheapest_product[1]}")
        else:
            print("📰 Ovo ne izgleda kao shop. Kratak rezime sajta:")
            print(summarize_text(page_text))

    except Exception as e:
        print("Greška:", e)

# pokretanje
url = input("Unesite link sajta: ")
analyze_website(url)

Unesite link sajta:  https://sr.wikipedia.org/sr-ec/Toni_Soprano


📰 Ovo ne izgleda kao shop. Kratak rezime sajta:
Најдужи прекид уследио је након што је Џуниор сазнао за Тонијеву терапију, па је Тони био приморан да је прекине и посаветује др Мелфи да се притаји на одређено време, или још боље да оде из града. Тони са др Мелфи прича о односима које има са својом породицом, пословним сарадницима, али и љубавницама које се у серији често смењују, затим о својим сновима, о својој мајци Ливији, која је непопустљиво песимистична и цинична, а у исто време тражи и презире помоћ. Спирс, Кевин Финерти, Бада Бинг (шифровано име за ФБИ ) Пол мушки Године 45 (оквирно) Занимање Консултант у фирми за управљање отпадом Сувласник месаре Satriale's и ноћног клуба Bada Bing Породица Џони Бој Сопрано (отац), Ливија Сопрано (мајка), Џенис Сопрано (сестра), Барбара Гиглионе (сестра), Корадо Џон Сопрано (стриц), Тони Блундето (рођак) Супруг(а) Кармела Сопрано Деца Медоу Сопрано (ћерка), Ентони Сопрано Млађи (син) Религија Католик Националност Американац Тумач Џејмс Гандол

TypeError: 'dict' object is not callable